# DAN 2 — Klasterovanje

Nastavljamo u istom notebook-u (ili u novoj sesiji — u tom slučaju prvo pokrenuti
ćelije iz Dana 1, ili učitati sačuvane fajlove iz `data/` i `output/` ispod).


In [ ]:
# Ucitavanje sacuvanih skupova iz Dana 1
# NAPOMENA: ovaj notebook se pokrece nezavisno (nova kernel sesija), pa ponovo
# uvozimo sve potrebne biblioteke i ucitavamo medjurezultate sacuvane na disku
# tokom Dana 1 (data/*.csv, output/*.pkl) - ne oslanja se na promenljive iz
# prethodnog notebook-a u memoriji.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import time
import os

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df_pca50 = pd.read_csv('../data/data_preprocessed_pca50.csv')
df_topvar = pd.read_csv('../data/data_preprocessed_topvar200.csv')

y_true = df_pca50['Tissue'].values
X_pca50 = df_pca50.drop(columns=['Tissue']).values
X_topvar = df_topvar.drop(columns=['Tissue']).values

print("PCA-50 shape:", X_pca50.shape)
print("TopVar-200 shape:", X_topvar.shape)


## 1. Skup atributa "Full"

Za pun skup (10935 standardizovanih atributa) ne čuvamo ceo CSV u repo (prevelik je),
pa ga ovde regenerišemo direktno iz originalnog `.arff` fajla + sačuvanog scaler-a,
radi potpune reproducibilnosti bez ručnog čuvanja ogromnih fajlova.


In [ ]:
from scipy.io import arff

data, meta = arff.loadarff('../data/OVA_Breast.arff')
df_raw = pd.DataFrame(data)
df_raw['Tissue'] = df_raw['Tissue'].str.decode('utf-8')

with open('../output/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

X_full = scaler.transform(df_raw.drop(columns=['ID_REF', 'Tissue']))
print("Full shape:", X_full.shape)

# Sanity check da se y_true poklapa sa onim iz PCA/TopVar skupova
assert (df_raw['Tissue'].values == y_true).all(), "Redosled instanci se ne poklapa!"
print("Redosled instanci potvrdjen - isti kao u PCA-50/TopVar-200 skupovima.")


In [ ]:
datasets = {
    'Full': X_full,
    'PCA-50': X_pca50,
    'TopVar-200': X_topvar,
}
for name, X in datasets.items():
    print(f"{name}: {X.shape}")


## 2. Određivanje optimalnog broja klastera (Elbow + Silhouette)

Za algoritme koji zahtevaju unapred zadat broj klastera (K-Means, Agglomerative, BIRCH)
koristimo Elbow metodu (inercija) kombinovanu sa Silhouette analizom, testirano na
PCA-50 skupu (reprezentativan i računski jeftin za ovu analizu).


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K_range = range(2, 11)
inertias = []
sils = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca50)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_pca50, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(K_range), inertias, 'o-', color='#378ADD')
axes[0].set_xlabel('Broj klastera (K)')
axes[0].set_ylabel('Inercija')
axes[0].set_title('Elbow metoda')

axes[1].plot(list(K_range), sils, 'o-', color='#D85A30')
axes[1].set_xlabel('Broj klastera (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette analiza')

plt.tight_layout()
plt.savefig('../visualizations/elbow_silhouette.png', dpi=150)
plt.show()

best_k = list(K_range)[np.argmax(sils)]
print(f"Optimalan K po Silhouette metrici: {best_k}")
print(f"Napomena: Silhouette raste skoro monotono do kraja testiranog opsega, sto je "
      f"cesta pojava kod HDLSS podataka. Testiramo K={best_k} kao 'fini' izbor i K=2 "
      f"kao 'grubi' izbor (motivisan binarnom prirodom Tissue atributa, iako se ne "
      f"koristi direktno u klasterovanju).")


Napomena o metodologiji: pošto Silhouette kriva raste do kraja testiranog opsega
(K=10), a to je karakteristično za visoko-dimenzione retke podatke gde ne postoji
jasan "lakat", testiramo **oba** K=10 (po formalnom kriterijumu) i K=2 (motivisano
prirodom problema — binarna klasa) radi potpunijeg poređenja, slično pristupu iz
referentnog rada (Farm-Ads, gde je testirano i K=10 i K=3).


---
### 🔵 GIT COMMIT — Dan 2
```bash
git add visualizations/elbow_silhouette.png
git commit -m "feat: elbow i silhouette analiza za odredjivanje broja klastera"
git push
```
---

## 3. Algoritmi klasterovanja

Primenjujemo **6 algoritama/konfiguracija** (prelazi minimum od 5 traženih uputstvom),
na sve **3 varijante skupa atributa** (Full, PCA-50, TopVar-200) — ukupno 18 kombinacija:

1. **K-Means (K=2)** — grub izbor, motivisan binarnom prirodom problema
2. **K-Means (K=10)** — fini izbor po Silhouette kriterijumu
3. **Agglomerative Clustering (Ward linkage, K=10)**
4. **Agglomerative Clustering (Complete linkage, K=10)**
5. **DBSCAN** — eps određen automatski preko k-distance grafika (90-ti percentil)
6. **BIRCH (K=10)**

Svi stohastički algoritmi koriste `random_state=42` radi reproducibilnosti.


In [ ]:
from sklearn.cluster import AgglomerativeClustering, DBSCAN, Birch
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                              calinski_harabasz_score, adjusted_rand_score,
                              normalized_mutual_info_score)

def evaluate_clustering(name, dataset_name, X, labels, y_true):
    """Racuna interne i eksterne metrike za jedan rezultat klasterovanja.
    Eksterne metrike (ARI, NMI) porede se sa Tissue labelom ISKLJUCIVO radi
    naknadne interpretacije - labela se nigde ne koristi kao ulaz modelu."""
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int(np.sum(labels == -1))
    mask = labels != -1

    if n_clusters < 2 or mask.sum() < 2:
        sil, db, ch = np.nan, np.nan, np.nan
    else:
        try:
            sil = silhouette_score(X[mask], labels[mask])
            db = davies_bouldin_score(X[mask], labels[mask])
            ch = calinski_harabasz_score(X[mask], labels[mask])
        except Exception:
            sil, db, ch = np.nan, np.nan, np.nan

    ari = adjusted_rand_score(y_true, labels)
    nmi = normalized_mutual_info_score(y_true, labels)

    return {
        'Algorithm': name, 'Dataset': dataset_name, 'K': n_clusters, 'Noise': n_noise,
        'Silhouette': sil, 'DaviesBouldin': db, 'CalinskiHarabasz': ch,
        'ARI_vs_Tissue': ari, 'NMI_vs_Tissue': nmi
    }


In [ ]:
# Recnik za cuvanje svih labela klastera (radi kasnije analize/vizuelizacije)
all_labels = {}
results = []

for ds_name, X in datasets.items():
    t0 = time.time()
    print(f"--- Obrada skupa: {ds_name} (oblik {X.shape}) ---")

    # 1. KMeans K=2
    km2 = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10).fit(X)
    all_labels[(ds_name, 'KMeans (K=2)')] = km2.labels_
    results.append(evaluate_clustering('KMeans (K=2)', ds_name, X, km2.labels_, y_true))

    # 2. KMeans K=10
    km10 = KMeans(n_clusters=10, random_state=RANDOM_STATE, n_init=10).fit(X)
    all_labels[(ds_name, 'KMeans (K=10)')] = km10.labels_
    results.append(evaluate_clustering('KMeans (K=10)', ds_name, X, km10.labels_, y_true))

    # 3. Agglomerative Ward
    agg_w = AgglomerativeClustering(n_clusters=10, linkage='ward').fit(X)
    all_labels[(ds_name, 'Agglomerative (Ward)')] = agg_w.labels_
    results.append(evaluate_clustering('Agglomerative (Ward)', ds_name, X, agg_w.labels_, y_true))

    # 4. Agglomerative Complete
    agg_c = AgglomerativeClustering(n_clusters=10, linkage='complete').fit(X)
    all_labels[(ds_name, 'Agglomerative (Complete)')] = agg_c.labels_
    results.append(evaluate_clustering('Agglomerative (Complete)', ds_name, X, agg_c.labels_, y_true))

    # 5. DBSCAN - eps procenjen preko k-distance grafika (90-ti percentil 5-nn rastojanja)
    nn = NearestNeighbors(n_neighbors=5).fit(X)
    distances, _ = nn.kneighbors(X)
    k_dist = np.sort(distances[:, -1])
    eps_guess = np.percentile(k_dist, 90)
    dbs = DBSCAN(eps=eps_guess, min_samples=5).fit(X)
    all_labels[(ds_name, 'DBSCAN')] = dbs.labels_
    results.append(evaluate_clustering(f'DBSCAN (eps={eps_guess:.2f})', ds_name, X, dbs.labels_, y_true))

    # 6. BIRCH
    birch = Birch(n_clusters=10, threshold=0.5).fit(X)
    all_labels[(ds_name, 'BIRCH')] = birch.labels_
    results.append(evaluate_clustering('BIRCH', ds_name, X, birch.labels_, y_true))

    print(f"  Zavrseno za {time.time()-t0:.1f}s")

results_df = pd.DataFrame(results)
print("\nSvi eksperimenti zavrseni:", len(results_df), "kombinacija")


In [ ]:
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)
results_df_sorted = results_df.sort_values('Silhouette', ascending=False)
results_df_sorted.round(3)


---
### 🔵 GIT COMMIT — Dan 2
```bash
git add notebooks/
git commit -m "feat: KMeans i Agglomerative na Full, PCA-50 i TopVar-200 skupovima"
git push
```
---

In [ ]:
# Cuvanje sirovih rezultata i labela za Dan 3
results_df.to_csv('../output/clustering_results.csv', index=False)

with open('../output/all_cluster_labels.pkl', 'wb') as f:
    pickle.dump(all_labels, f)

print("Sacuvano: output/clustering_results.csv, output/all_cluster_labels.pkl")
print("\nDAN 2 zavrsen. Spremno za Dan 3 - evaluacija i vizuelizacija.")


---
### 🔵 GIT COMMIT — Dan 2
```bash
git add output/clustering_results.csv output/all_cluster_labels.pkl
git commit -m "feat: DBSCAN i BIRCH runovi, cuvanje svih rezultata klasterovanja"
git push
```
---